# Agriculture and income: preparing World Bank data

This notebook prepares one clean table from three World Bank files. The result can be used to explore how GDP per person relates to agriculture in different countries.

**Question:** Do countries with more agricultural land or a larger agriculture share of GDP tend to have lower GDP per person?

The notebook checks years from 1960 to 2025. Complete records start in 1961 because the agricultural-land data has no 1960 values. 


In [1]:
from pathlib import Path

import pandas as pd

data_folder = Path.cwd() / "phydata_capstone data"
if not data_folder.is_dir():
    data_folder = Path.cwd().parent / "phydata_capstone data"
if not data_folder.is_dir():
    raise FileNotFoundError("Put the 'phydata_capstone data' folder in the project folder.")


gdp_file = next(data_folder.glob(
    "API_NY.GDP.PCAP.CD_*/API_NY.GDP.PCAP.CD_*.csv"
))
agri_gdp_file = next(data_folder.glob(
    "API_NV.AGR.TOTL.ZS_*/API_NV.AGR.TOTL.ZS_*.csv"
))
agri_land_file = next(data_folder.glob(
    "API_AG.LND.AGRI.ZS_*/API_AG.LND.AGRI.ZS_*.csv"
))
country_info_file = next(data_folder.glob(
    "API_NY.GDP.PCAP.CD_*/Metadata_Country_*.csv"
))


first_year = 1960       
last_year = 2025        
min_years = 10          

print("Data folder:", data_folder)
print("Years:", first_year, "to", last_year)

Data folder: C:\Users\MARTIN\capstone-martin\phydata_capstone data
Years: 1960 to 2025


## Keep country observations

In [2]:
country_info = pd.read_csv(country_info_file)

real_countries = country_info[country_info["Region"].notna()]      # keep rows that have a Region
real_country_codes = list(real_countries["Country Code"])           # make a list of their codes

print("Number of real countries and territories:", len(real_country_codes))

Number of real countries and territories: 217


## Prepare a reusable cleaning function

In [3]:

def load_and_clean(file_path, new_column_name):

   
    table = pd.read_csv(file_path, skiprows=4)

    table = table.drop(columns=["Indicator Name", "Indicator Code"])
    empty_columns = [name for name in table.columns if name.startswith("Unnamed")]
    table = table.drop(columns=empty_columns)


    table = table.melt(
        id_vars=["Country Name", "Country Code"],   
        var_name="Year",                            
        value_name=new_column_name                  
    )

    
    table = table.dropna()
    table["Year"] = table["Year"].astype(int)

    #  Keep only the years we want.
    table = table[(table["Year"] >= first_year) & (table["Year"] <= last_year)]

    #  Keep only real countries (not groups like "World").
    table = table[table["Country Code"].isin(real_country_codes)]

    #  Remove any accidental duplicate rows.
    table = table.drop_duplicates(subset=["Country Code", "Year"])

    return table

print("Cleaning function is ready.")

Cleaning function is ready.


## Load and check the three tables

In [4]:
gdp      = load_and_clean(gdp_file,      "GDP_per_capita")
agri_gdp = load_and_clean(agri_gdp_file, "Agriculture_GDP")
agri_land = load_and_clean(agri_land_file, "Agriculture_Land")

# Check how many rows and countries each table has
print("GDP per capita:   ", len(gdp),      "rows,", gdp["Country Code"].nunique(),      "countries")
print("Agriculture % GDP:", len(agri_gdp), "rows,", agri_gdp["Country Code"].nunique(), "countries")
print("Agricultural land:", len(agri_land), "rows,", agri_land["Country Code"].nunique(), "countries")

# Look at the first 5 rows of one table
gdp.head()

GDP per capita:    11764 rows, 214 countries
Agriculture % GDP: 9088 rows, 206 countries
Agricultural land: 12121 rows, 210 countries


,Country Name,Country Code,Year,GDP_per_capita
9,Argentina,ARG,1960,778.251707
13,Australia,AUS,1960,1813.431099
14,Austria,AUT,1960,939.914815
16,Burundi,BDI,1960,70.905100
17,Belgium,BEL,1960,1290.286072


## Join the tables

In [5]:
# We match rows that have the same Country and Year.
# "inner" means we only keep country (years that appear in ALL three tables).

match_on = ["Country Name", "Country Code", "Year"]

df = gdp.merge(agri_gdp, on=match_on, how="inner")
df = df.merge(agri_land, on=match_on, how="inner")

print("Merged table:", df.shape[0], "rows,", df["Country Code"].nunique(), "countries")
df.head()

Merged table: 8510 rows, 203 countries


,Country Name,Country Code,Year,GDP_per_capita,Agriculture_GDP,Agriculture_Land
0,Benin,BEN,1961,92.085848,46.193786,12.544874
1,Burkina Faso,BFA,1961,72.242379,38.307008,32.949561
2,Bangladesh,BGD,1961,90.368579,57.988236,72.827841
3,Brazil,BRA,1961,231.564063,0.000000,18.779178
4,Botswana,BWA,1961,62.322401,41.925895,45.877225


## Check and clean the joined data

In [6]:
# Remove countries that have too few years of data.
years_per_country = df.groupby("Country Code")["Year"].transform("count")
removed_names = df[years_per_country < min_years]["Country Name"].unique()
print("Removed for having fewer than", min_years, "years:", list(removed_names))
df = df[years_per_country >= min_years]

# Remove impossible values.
#     Percentages must be between 0 and 100. GDP per capita must be above 0.
good_rows = (
    (df["GDP_per_capita"] > 0)
    & (df["Agriculture_GDP"] >= 0) & (df["Agriculture_GDP"] <= 100)
    & (df["Agriculture_Land"] >= 0) & (df["Agriculture_Land"] <= 100)
)
print("Impossible rows removed:", (~good_rows).sum())
df = df[good_rows]

#  Sort the table neatly and number the rows again from 0.
df = df.sort_values(["Country Name", "Year"])
df = df.reset_index(drop=True)


print("FINAL DATA:", df.shape[0], "rows,", df["Country Code"].nunique(), "countries,",
      "years", df["Year"].min(), "to", df["Year"].max())
print("Missing values:", df.isna().sum().sum(), "(should be 0)")
df.head()

Removed for having fewer than 10 years: ['South Sudan', 'San Marino', 'Channel Islands', 'Liechtenstein']
Impossible rows removed: 0
FINAL DATA: 8483 rows, 199 countries, years 1961 to 2023
Missing values: 0 (should be 0)


,Country Name,Country Code,Year,GDP_per_capita,Agriculture_GDP,Agriculture_Land
0,Afghanistan,AFG,2002,178.954088,38.627892,57.939684
1,Afghanistan,AFG,2003,198.871116,37.418855,58.083805
2,Afghanistan,AFG,2004,221.763654,29.721067,58.151266
3,Afghanistan,AFG,2005,254.184249,31.114855,58.134400
4,Afghanistan,AFG,2006,274.218554,28.635969,58.123668


## Save the cleaned data

In [7]:
output_file = data_folder / "capstone_clean.csv"
df.to_csv(output_file, index=False)
print("Saved to:", output_file)

Saved to: C:\Users\MARTIN\capstone-martin\phydata_capstone data\capstone_clean.csv
